# Part 4 — Real-Time Analysis with Spark Structured Streaming
**Project:** Financial Forecasting Frontier: Distributed ML  
**Contribution:** Individual — Suman

## Summary and scope
Replay small chunks of the held-out banking marketing dataset through actual incremental Spark queries. Every event receives a prediction from Part 3's saved pre-call model. Track balances and call durations by job and over 10-second and one-minute windows. Event timestamps are synthetic; source duration describes a marketing call, not transaction latency. No actual fraud labels or payment events are supplied.

## Q1. Stream processing and aggregation
JSON chunk files contain age, job, balance, duration, y and all other source fields. The file source reads one file per microbatch in modification-time order. A complete-mode stateful job-category aggregation updates count, average balance and average call duration after each batch. foreachBatch displays and saves the actual outputs, and temporary views make the retained results inspectable; durable production sinks would be required in production.


In [1]:
import sys
sys.path.insert(0,'/workspace')
from banking.common import *
from banking.streaming import run
prepare_dirs()
spark=session('Banking-Structured-Streaming')
result=run(spark)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/21 04:44:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/10/21 04:47:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/10/21 04:48:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


bank_predictions batch 0 output rows 3


...

+------+-------+-----------+
|window|records|avg_balance|
+------+-------+-----------+
+------+-------+-----------+



bank_window_1m batch 6 output rows 1


+------------------------------------------+-------+-----------+
|window                                    |records|avg_balance|
+------------------------------------------+-------+-----------+
|{2026-01-01 00:01:00, 2026-01-01 00:02:00}|5      |1530.8     |
+------------------------------------------+-------+-----------+



bank_window_1m : 2 output rows


+------------------------------------------+-------+-----------+
|window                                    |records|avg_balance|
+------------------------------------------+-------+-----------+
|{2026-01-01 00:00:00, 2026-01-01 00:01:00}|8      |3649.125   |
|{2026-01-01 00:01:00, 2026-01-01 00:02:00}|5      |1530.8     |
+------------------------------------------+-------+-----------+

Late rows dropped: {'bank_predictions': 0, 'bank_job_live': 0, 'bank_window_10s': 1, 'bank_window_1m': 1}


## Q2. Real-time model predictions
The saved PipelineModel applies identical imputation, indexing, encoding, scaling and classification to each arrival. It never refits on streamed data. Replay uses only held-out records; label y is included for inspection, never as a feature. Prediction output retains event ID, arrival batch and positive-class probability.


In [2]:
spark.table('bank_predictions').orderBy('arrival_batch','event_id').show(25,truncate=False)

+---------+-------------------+-------------+---+----------+------------------------+
|event_id |event_time         |arrival_batch|y  |prediction|subscription_probability|
+---------+-------------------+-------------+---+----------+------------------------+
|event_000|2026-01-01 00:00:00|1            |no |0.0       |0.09437751004016064     |
|event_001|2026-01-01 00:00:01|1            |no |0.0       |0.09437751004016064     |
|event_002|2026-01-01 00:00:02|1            |no |0.0       |0.09437751004016064     |
|event_003|2026-01-01 00:00:12|2            |no |0.0       |0.09437751004016064     |
|event_004|2026-01-01 00:00:13|2            |no |0.0       |0.13636363636363635     |
|event_005|2026-01-01 00:00:14|2            |no |0.0       |0.09437751004016064     |
|event_006|2026-01-01 00:00:07|3            |no |0.0       |0.09437751004016064     |
|event_007|2026-01-01 00:00:08|3            |no |1.0       |0.7605633802816901      |
|event_008|2026-01-01 00:01:30|4            |no |0.0  

## Q3. Window operations and trend analysis
Compute event-time windows of 10 seconds and one minute with a 20-second watermark. Append output contains finalized windows only. A later event advances the watermark; the last windows remain open rather than being treated as complete prematurely. These chosen time gaps demonstrate changing counts and averages; they do not reveal real historical banking trends.


In [3]:
for name in ['bank_job_live','bank_window_10s','bank_window_1m']:
    print(name)
    spark.table(name).orderBy('job' if name=='bank_job_live' else 'window').show(truncate=False)
print('First three source batches, including in-order and out-of-order arrivals:')
for event in result['events'][:8]:print(event['event_id'],event['event_time'],event['arrival_batch'])


bank_job_live


+-----------+-------+-----------------+------------------+
|job        |records|avg_balance      |avg_call_seconds  |
+-----------+-------+-----------------+------------------+
|admin.     |4      |1512.5           |629.75            |
|blue-collar|2      |2367.5           |204.0             |
|management |3      |6357.666666666667|118.33333333333333|
|retired    |2      |605.5            |248.0             |
|services   |2      |0.0              |332.0             |
|technician |3      |5083.666666666667|108.33333333333333|
|unemployed |1      |0.0              |108.0             |
+-----------+-------+-----------------+------------------+

bank_window_10s


+------------------------------------------+-------+-----------------+
|window                                    |records|avg_balance      |
+------------------------------------------+-------+-----------------+
|{2026-01-01 00:00:00, 2026-01-01 00:00:10}|5      |2885.8           |
|{2026-01-01 00:00:10, 2026-01-01 00:00:20}|3      |4921.333333333333|
|{2026-01-01 00:01:30, 2026-01-01 00:01:40}|5      |1530.8           |
+------------------------------------------+-------+-----------------+

bank_window_1m


+------------------------------------------+-------+-----------+
|window                                    |records|avg_balance|
+------------------------------------------+-------+-----------+
|{2026-01-01 00:00:00, 2026-01-01 00:01:00}|8      |3649.125   |
|{2026-01-01 00:01:00, 2026-01-01 00:02:00}|5      |1530.8     |
+------------------------------------------+-------+-----------+

First three source batches, including in-order and out-of-order arrivals:
event_000 2026-01-01T00:00:00+00:00 1
event_001 2026-01-01T00:00:01+00:00 1
event_002 2026-01-01T00:00:02+00:00 1
event_003 2026-01-01T00:00:12+00:00 2
event_004 2026-01-01T00:00:13+00:00 2
event_005 2026-01-01T00:00:14+00:00 2
event_006 2026-01-01T00:00:07+00:00 3
event_007 2026-01-01T00:00:08+00:00 3


## Q4. Late and out-of-order data
Batch 3 contains seconds 7 and 8 after batch 2 reached second 14: these events are inside the 20-second tolerance and update their earlier window. Batch 6 contains second 4 after batches 4 and 5 advanced event time and finalized that old window. Both window queries must now reject the record. The record at second 4 still receives a prediction because the prediction query does not discard arrivals by event time. Watermark propagation and eviction can lag a microbatch, so being beyond the configured delay does not guarantee immediate rejection. The progress log measures actual drops rather than assuming the demonstration worked.


In [4]:
print('Actual late-row counts:',result['dropped_by_watermark'])
for name in ['bank_window_10s','bank_window_1m']:
    print(name)
    for progress in result['queries'][name]['progress']:
        print('batch',progress['batchId'],'input',progress['numInputRows'],'event time',progress.get('eventTime'),'state',progress.get('stateOperators'))
spark.stop()


Actual late-row counts: {'bank_predictions': 0, 'bank_job_live': 0, 'bank_window_10s': 1, 'bank_window_1m': 1}
bank_window_10s
batch 0 input 3 event time {'avg': '2026-01-01T00:00:01.000Z', 'max': '2026-01-01T00:00:02.000Z', 'min': '2026-01-01T00:00:00.000Z', 'watermark': '1970-01-01T00:00:00.000Z'} state [{'operatorName': 'stateStoreSave', 'numRowsTotal': 1, 'numRowsUpdated': 1, 'allUpdatesTimeMs': 526, 'numRowsRemoved': 0, 'allRemovalsTimeMs': 542, 'commitTimeMs': 3156, 'memoryUsedBytes': 1256, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 4, 'numStateStoreInstances': 4, 'customMetrics': {'loadedMapCacheHitCount': 0, 'loadedMapCacheMissCount': 0, 'stateOnCurrentVersionSizeBytes': 680}}]
batch 1 input 3 event time {'avg': '2026-01-01T00:00:13.000Z', 'max': '2026-01-01T00:00:14.000Z', 'min': '2026-01-01T00:00:12.000Z', 'watermark': '2025-12-31T23:59:42.000Z'} state [{'operatorName': 'stateStoreSave', 'numRowsTotal': 2, 'numRowsUpdated': 1, 'allUpdatesTimeMs': 545, 'numRowsRem

## Conclusion and production extensions
All arrivals receive predictions; job summaries update incrementally; window results show the distinct treatment of acceptable and excessive lateness. Replay directories and checkpoints are unique per run to avoid accidentally reusing old state. A production system would use Kafka or an equivalent durable source, authenticated infrastructure, a durable output sink, idempotent event IDs, monitoring, schema validation and a reviewed model release process. Watermark tolerance trades completeness for bounded state.
